<div style="padding:10px 0; text-align:left; color: red;">
<h1>BNPL Instant Credit Risk Engine</h1>
</div>

<div style="border-top:2px solid gray; border-bottom:2px solid gray; padding:0px 0px 7px 0px; text-align:left;">
<h3>Introduction</h3>
</div>

**Project Description:**  

This project helps **Buy-Now-Pay-Later (BNPL) companies** decide **quickly and safely** whether to approve a loan for a customer.

**How it works:**

1. The system looks at **past loan data** (who paid on time, who didn’t).  
2. It uses a **logistic regression model** (a type of smart calculator) to estimate the **chance that a customer will default** (fail to pay back).  
3. For loans that were **rejected in the past**, it can still predict what would have happened (**reject inference**).  
4. The model predicts a **probability of default**, then uses a **business threshold** to decide:  
   - **Approve** if risk is low  
   - **Reject** if risk is high  
5. It can **simulate profit**, showing how much money the company would make or lose depending on the threshold.  
6. The model can run in **real-time** via a web API, so BNPL platforms can instantly score new loan applications.

**Goal:**  

- Approve **safe loans faster**  
- Reduce **losses from defaults**  
- Make **data-driven decisions** without manual checking

**In short:**  
It’s like a **smart credit assistant** that predicts if a customer will pay or not, helping BNPL companies lend money safely and profitably.

<div style="border-top:2px solid gray; border-bottom:2px solid gray; padding:0px 0px 7px 0px; text-align:left;">
<h3>Dataset</h3>
</div>

#### 🗂 Dataset Used: LendingClub Accepted & Rejected Loans

**Dataset Description:**  

We use the **LendingClub loan data** which contains **historical loan applications** from 2007 to 2018. The dataset is split into two parts:  

1. **Accepted Loans:** Applications that were **approved** by LendingClub.  
   - Contains detailed loan and borrower information such as loan amount, term, interest rate, income, credit history, FICO score, and more.  
   - Includes the final status of each loan: **Fully Paid** or **Charged Off** (defaulted).  

2. **Rejected Loans:** Applications that were **rejected** by LendingClub.  
   - Used for **reject inference**, to predict how rejected applicants might have performed if they had been approved.  

**Why this dataset is used:**  

- Provides **real-world historical data** to train a credit risk model.  
- Allows us to predict **default probability** for new BNPL applications.  
- Supports **risk assessment**, profit simulation, and business threshold decisions.  

**Dataset Link:**  
[Click here to access the dataset on Kaggle](https://www.kaggle.com/datasets/denychaen/lending-club-loans-rejects-data)  

**Important Note:**  
- The dataset includes both **numerical and categorical features**.  
- Some columns may have **missing values**, which are handled during preprocessing.  
- Using both accepted and rejected loans allows the model to **learn from a wider range of applicant profiles**.

#### 📊 Feature Explanation

##### Features We Use for Prediction:

| Feature | What It Is | Good = | Bad = |
|---------|-----------|--------|-------|
| **loan_amnt** | How much money borrowed | Small amount | Large amount |
| **int_rate** | Interest rate (%) | Low % | High % |
| **annual_inc** | Yearly income | High income | Low income |
| **dti** | Debt-to-income ratio | Low % | High % |
| **fico_range_low** | Credit score (lower bound) | High score (700+) | Low score (sub-600) |
| **loan_status** | What we predict | "Fully Paid" | "Charged Off" |

##### What We Want to Predict:

- **GOOD (0)** = "Fully Paid" - Borrower repaid the loan ✅
- **BAD (1)** = "Charged Off" - Borrower defaulted, bank lost money ❌

##### Simple Rule:

**Safer loans:** Low amount + Low interest + High income + Low DTI + High FICO

**Riskier loans:** High amount + High interest + Low income + High DTI + Low FICO

<div style="border-top:2px solid gray; border-bottom:2px solid gray; padding:0px 0px 7px 0px; text-align:left;">
<h3>Implementation and Coding</h3>
</div>

In [1]:
# ====================================================================================
# 0. Suppres Warnings
# ====================================================================================

In [2]:
# Fix Joblib Warning (CPU Core Detection)
# ------------------------------------------------------------------------------------

import os               # Gives access to computer settings
import multiprocessing  # Helps detect how many CPU cores you have

# This line tells joblib (used by scikit-learn) how many CPU cores to use
# Without this, joblib tries to auto-detect and sometimes fails on Windows
# The failure causes a warning (harmless but annoying)
# 
# What happens here:
# 1. multiprocessing.cpu_count() - Detects your actual CPU cores (e.g., 4, 8, 16)
# 2. str() - Converts the number to text (environment variables need text)
# 3. os.environ["LOKY_MAX_CPU_COUNT"] = ... - Saves this number for joblib to read
# 
# When joblib sees this number, it skips auto-detection and uses your number
# Result: No warning! ✅
os.environ["LOKY_MAX_CPU_COUNT"] = str(multiprocessing.cpu_count())

# Optional: Print confirmation
print(f"✅ Joblib will use {multiprocessing.cpu_count()} CPU cores (no auto-detection warning)")


# Backup Solution: Just hide the warning
# ------------------------------------------------------------------------------------

import warnings                                # Import warning handling module
warnings.filterwarnings("ignore",              # Don't show the warning
                        category=UserWarning,  # Only ignore UserWarning type
                        module="joblib")       # Only from joblib module

✅ Joblib will use 8 CPU cores (no auto-detection warning)


In [3]:
# ====================================================================================
# 1. Import Libraries
# ====================================================================================

In [4]:
import pandas as pd                                            # Library for data manipulation (DataFrame, CSV read/write)
import numpy as np                                             # Library for numerical operations (arrays, math operations)
import csv                                                     # Built-in module for reading and writing CSV files (alternative to pandas for simple operations)
from sklearn.model_selection import train_test_split           # Split dataset into training and testing
from sklearn.preprocessing import StandardScaler               # Standardize numeric features (mean=0, std=1)
from sklearn.linear_model import (SGDClassifier,               # Linear classifier trained via SGD, efficient for large datasets and online learning
                                  LogisticRegression)          # Linear model for classification using logistic function (supports L1/L2 regularization)
from sklearn.ensemble import (HistGradientBoostingClassifier,  # Fast gradient boosting that builds trees sequentially (supports incremental learning via warm_start)
                              VotingClassifier,                # Combines multiple models by majority vote to improve prediction stability
                              RandomForestClassifier )         # Ensemble of many decision trees that vote on final prediction (bagging method)
from sklearn.tree import DecisionTreeClassifier                # Tree-based model that splits data into branches based on feature values (easy to interpret)
from sklearn.svm import SVC                                    # Support Vector Classifier that finds the best boundary between classes (works well for small/medium data)
from sklearn.calibration import CalibratedClassifierCV         # Calibrate predicted probabilities for accuracy
from sklearn.metrics import (accuracy_score,                   # % of correct predictions (good for balanced data)
                             precision_score,                  # Of loans predicted BAD, how many were actually BAD?
                             recall_score,                     # Of actual BAD loans, how many did we catch?
                             f1_score,                         # Balance between precision and recall (harmonic mean)
                             roc_auc_score,                    # How well model separates GOOD from BAD loans (0.5=random, 1.0=perfect)
                             classification_report,            # Complete table of precision, recall, f1 for all classes
                             confusion_matrix)                 # Table showing: TP, TN, FP, FN counts
import matplotlib.pyplot as plt                                # Plotting library
import pickle                                                  # Save/load Python objects (model, scaler)
import joblib                                                  # Save and load trained models (more efficient than pickle for scikit-learn)
from datetime import datetime                                  # Get current date and time (useful for file naming and logging)
import time                                                    # Provides time-related functions (sleep, timing, delays) - used here to pause execution for 5 seconds before falling back to Phase 1
from fastapi import FastAPI                                    # Web framework to create API endpoints
from pydantic import BaseModel                                 # Define structured input for API with type validation
import uvicorn                                                 # Server to run FastAPI application

In [5]:
# ====================================================================================
# 2. Load Accepted Loans Dataset
# ====================================================================================

In [6]:
# Raw string path to LendingClub accepted loan applications data (2007-2019 Q3)
DATA_PATH = r"D:\Data\LendingClubLoanData\appl_accepted_20072019Q3.csv"  

In [7]:
# Load only first specific rows to preview data structure and reduce memory usage
data = pd.read_csv(DATA_PATH, nrows=5)

# Display the first 5 rows of the loaded dataframe for visual inspection
data.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# ====================================================================================
# 3. Incremental Learning with Memory-Efficient Chunking
# ====================================================================================

In [9]:
# Number of rows to load at once (prevents RAM overload)
CHUNK_SIZE = 50000

In [10]:
# ====================================================================================
# 3.1. Select only required columns
# ====================================================================================

In [11]:
USE_COLS = [
    "loan_amnt",      # Loan amount (features)
    "int_rate",       # Interest rate (features)
    "annual_inc",     # Annual income (features)
    "dti",            # Debt-to-income ratio (features)
    "fico_range_low", # Credit score lower bound (features)
    "loan_status"     # Target: what we want to predict
]

In [12]:
# ====================================================================================
# Memory-efficient dtypes
# ====================================================================================

In [13]:
DTYPES = {
    "loan_amnt": "float32",      # 32-bit float (saves memory vs float64)
    "int_rate": "string",        # Keep as text (string) because values have "%" (e.g., "10.5%")
    "annual_inc": "float32",     # 32-bit float
    "dti": "float32",            # 32-bit float
    "fico_range_low": "float32", # 32-bit float
    "loan_status": "category"    # Stores repeated text as codes (e.g., "Fully Paid"=0, "Charged Off"=1)
}

In [14]:
# ====================================================================================
# 3.2. Initialize incremental learning model(s)
# ====================================================================================

In [15]:
# ====================================================================================
# 3.2.1. SGD Classifier (Logistic Regression)
# ====================================================================================
# SGDClassifier = Stochastic Gradient Descent Classifier
# Works with chunks (partial_fit) without loading all data at once
# Best for: Memory efficiency, fast training, strong baseline
# ------------------------------------------------------------------------------------

In [16]:
# Uses partial_fit() for incremental/online learning
model1 = SGDClassifier(
    loss="log_loss",           # Makes it work like Logistic Regression (outputs probabilities)
    random_state=42,           # Fixes random numbers for reproducible results
    penalty="l2",              # L2 regularization (prevents overfitting)
    alpha=0.0001,              # Regularization strength (higher = more regularization)
    max_iter=1000,             # Maximum iterations per chunk
    tol=1e-3,                  # Stopping tolerance (stop when improvement < this)
    learning_rate="optimal",   # Adaptive learning rate schedule
    eta0=0.0,                  # Initial learning rate (0 = auto by 'optimal')
    #class_weight="balanced",  # 🔑 Balances class weights for imbalanced data (bad loans get higher importance) (NOT WORKING FOR PARTIAL_FIT)
    class_weight={0: 1.0, 1: 4.56}
)
# ----------------------------------------------------------------------------
# 🔑 Class Weights Explanation: class_weight={0: 1.0, 1: 4.56}
# ----------------------------------------------------------------------------
#
# WHAT IT DOES:
#   Tells the model how much importance to give each class during training.
#
# THE VALUES:
#   0 (Good Loan)  -> weight 1.0  (normal importance)
#   1 (Bad Loan)   -> weight 4.56 (4.56x MORE important than Good loans)
#
# WHY 4.56?
#   Your data is imbalanced: 82% Good loans, 18% Bad loans
#   Ratio = 82 / 18 = 4.56 (balances the scale)
#
# WITHOUT class weights:
#   Model sees more Good loans → learns to predict "GOOD" for everything
#   Result: ❌ Misses ALL bad loans
#
# WITH class weights {0:1.0, 1:4.56}:
#   Good: 82 loans × 1.0 = 82 effective weight
#   Bad:  18 loans × 4.56 = 82 effective weight
#   Result: ✅ Both classes equal → model catches bad loans
#
# NOTE: "balanced" does the same thing automatically but does NOT work with
#       partial_fit(). So we use manual weights: {0: 1.0, 1: 4.56}
#
# ----------------------------------------------------------------------------

# Possible target values (0 = Good Loan, 1 = Bad Loan/Charged Off)
# int8 uses only 1 byte per value (very memory efficient)
classes = np.array([0, 1], dtype=np.int8)

print("✅ Approach 1 Model Ready: SGDClassifier (Logistic Regression)")
print(f"   Model type: {type(model1).__name__}")
print(f"   Loss function: {model1.loss}")
print(f"   Class weights: BALANCED (handles imbalanced data)")
print(f"   Classes: {classes}")

✅ Approach 1 Model Ready: SGDClassifier (Logistic Regression)
   Model type: SGDClassifier
   Loss function: log_loss
   Class weights: BALANCED (handles imbalanced data)
   Classes: [0 1]


In [17]:
# Regularization Explanation
# ============================================================================
#
# WHAT IS REGULARIZATION?
#   Prevents the model from "memorizing" the data instead of "learning" patterns.
#   Like a teacher saying "show your work" - prevents shortcuts and guessing.
#
# WHY NEEDED?
#   Without regularization: Model learns noise + exceptions → fails on new data
#   With regularization:    Model learns general rules → works on new data
#
# ANALOGY:
#   Studying for an exam:
#   - WITHOUT regularization: Memorizing exact answers (fails if questions change)
#   - WITH regularization:    Understanding concepts (passes any variation)
#
# ============================================================================

# penalty="l2" - Type of regularization (Ridge / L2)
#   - Shrinks large coefficients toward zero (but never to zero)
#   - Prevents any single feature from dominating the prediction
#   - "l2" is the most common and works well for most cases
#   - Alternative "l1" would completely remove unimportant features
#   - Best for: When all features might be useful (our case)
#
# alpha=0.0001 - Regularization strength (controls how much regularization)
#   - alpha = 0.0    → NO regularization (model memorizes, overfits badly)
#   - alpha = 0.0001 → LOW regularization (good balance - we use this)
#   - alpha = 0.01   → MEDIUM regularization (safer, may underfit)
#   - alpha = 1.0    → HIGH regularization (too strict, model underfits)
#   - alpha = 10.0   → VERY HIGH (model becomes too simple, loses patterns)
#
# VISUAL EXAMPLE:
#   Without regularization (alpha=0):
#     Model: "Loan amount $10,001 → BAD, but $10,000 → GOOD" (memorizes random noise)
#   
#   With regularization (alpha=0.0001):
#     Model: "Higher loan amount = slightly higher risk" (learns general rule)
#
# SIMPLE RULE:
#   - Model performing poorly on test data? → Increase alpha (more regularization)
#   - Model too simple (underfitting)?      → Decrease alpha (less regularization)
#
# ============================================================================

In [18]:
# tol=1e-3 - Stopping Tolerance and Improvement
# ============================================================================
#
# WHAT IT DOES:
#   Tells the model when to stop training because "good enough"
#
# IMPROVEMENT = How much the model's ERROR decreased in the last step
#
# HOW IT WORKS:
#   Each training step: "Did my error go down enough to continue?"
#   If improvement < tol → STOP (not worth continuing)
#   If improvement ≥ tol → CONTINUE (keep learning)
#
# EXAMPLE:
#   Step 1: Error = 0.50
#   Step 2: Error = 0.30 → Improvement = 0.20 (big) → CONTINUE
#   Step 3: Error = 0.29 → Improvement = 0.01 (big) → CONTINUE  
#   Step 4: Error = 0.2901 → Improvement = 0.0001 (small)
#   
#   tol=1e-3 = 0.001
#   Improvement (0.0001) < tol (0.001) → STOP training
#
# ANALOGY:
#   Studying for a test: Stop when your score improves by less than 0.1%
#
# COMMON VALUES:
#   tol=1e-6 (0.000001) → Very strict, trains too long
#   tol=1e-4 (0.0001)   → Strict, for high accuracy
#   tol=1e-3 (0.001)    → DEFAULT, good balance ✅
#   tol=1e-2 (0.01)     → Lenient, stops early
#
# SIMPLE RULE:
#   Training too slow? → INCREASE tol (e.g., 1e-2)
#   Not accurate enough? → DECREASE tol (e.g., 1e-4)
#
# ============================================================================
#
# tol=1e-3,  # Stopping tolerance - stop when improvement is less than 0.001 (0.1%)

In [19]:
# ====================================================================================
# 3.2.2. HistGradientBoostingClassifier
# ====================================================================================
# Hist Gradient Boosting Classifier
# HistGradientBoostingClassifier = Fast gradient boosting with histogram binning
# Uses warm_start for incremental learning (adds trees each chunk)
# Best for: Better accuracy, handles non-linear patterns
# ------------------------------------------------------------------------------------

In [20]:
# Uses warm_start=True to add more trees incrementally (not partial_fit)
model2 = HistGradientBoostingClassifier(
    warm_start=True,           # Enables incremental learning (adds trees, doesn't retrain)
    max_iter=50,               # Start with 50 trees (will increase with each chunk)
    max_depth=10,              # Maximum depth of each tree (prevents overfitting)
    learning_rate=0.1,         # Step size shrinkage (lower = more robust)
    random_state=42,           # Reproducible results
    n_iter_no_change=10,       # Early stopping if no improvement for 10 iterations
    validation_fraction=0.1,   # Use 10% of data for validation
    min_samples_leaf=20,       # Minimum samples per leaf (prevents overfitting)
    verbose=1                  # Show training progress
)

# Possible target values (0 = Good Loan, 1 = Bad Loan/Charged Off)
# int8 uses only 1 byte per value (very memory efficient)
classes = np.array([0, 1], dtype=np.int8)

print("✅ Approach 2 Model Ready: HistGradientBoostingClassifier")
print(f"   Model type: {type(model2).__name__}")
print(f"   Initial trees: {model2.max_iter}")
print(f"   Max depth: {model2.max_depth}")
print(f"   Classes: {classes}")
print("   ⚠️  Note: Training loop needs warm_start logic (not partial_fit)")

✅ Approach 2 Model Ready: HistGradientBoostingClassifier
   Model type: HistGradientBoostingClassifier
   Initial trees: 50
   Max depth: 10
   Classes: [0 1]
   ⚠️  Note: Training loop needs warm_start logic (not partial_fit)


In [21]:
# ====================================================================================
# 3.2.3. Ensemble Voting Classifier (Sample-Based Only)
# ====================================================================================
# ⚠️  WARNING: Does NOT support partial_fit or warm_start
# ⚠️  Use ONLY if you can load a sample into RAM (200k+ rows)
#
# IMPORTANT - READ BEFORE USING:
# ------------------------------------------------------------------------------------
# This ensemble model CANNOT be used with your chunked training loop!
# It requires loading a sample of data into RAM and using .fit()
# 
# To use this model, replace your entire training loop with:
#   1. Load sample: df = pd.read_csv(FILE_PATH, nrows=200000)
#   2. Preprocess: X, y = preprocess_chunk(df)
#   3. Train: model.fit(X, y)
#
# DO NOT use with partial_fit() - it will fail!
# ------------------------------------------------------------------------------------

In [22]:
# Individual models
log_reg = LogisticRegression(
    max_iter=1000,           # More iterations for convergence
    random_state=42,         # Reproducible results
    class_weight="balanced"  # Handle imbalanced classes
)

decision_tree = DecisionTreeClassifier(
    max_depth=10,             # Limit depth to prevent overfitting
    random_state=42,          # Reproducible results
    class_weight="balanced"   # Handle imbalanced classes
)

random_forest = RandomForestClassifier(
    n_estimators=100,         # Number of trees in the forest
    max_depth=10,             # Limit depth for memory efficiency
    random_state=42,          # Reproducible results
    class_weight="balanced",  # Handle imbalanced classes
    n_jobs=-1                 # Use all CPU cores
)

svm = SVC(
    probability=True,        # Required for soft voting (probability estimates)
    random_state=42,         # Reproducible results
    kernel="rbf",            # Radial basis function kernel (good for non-linear data)
    class_weight="balanced"  # Handle imbalanced classes
)

# Combine all models into one ensemble
# Voting = each model "votes" on the prediction
model3 = VotingClassifier(
    estimators=[
        ("logistic", log_reg),             # Logistic Regression
        ("decision_tree", decision_tree),  # Decision Tree
        ("random_forest", random_forest),  # Random Forest
        ("svm", svm)                       # Support Vector Machine
    ],
    voting="soft",        # 'soft' = uses probability scores (better than 'hard' voting)
    weights=[1, 1, 2, 1]  # Give more weight to Random Forest (usually performs best)
)

# Possible target values (0 = Good Loan, 1 = Bad Loan/Charged Off)
# int8 uses only 1 byte per value (very memory efficient)
classes = np.array([0, 1], dtype=np.int8)

print("✅ Approach 3 Model Ready: VotingClassifier Ensemble")
print(f"   Models in ensemble: {len(model3.estimators)}")
for name, est in model3.estimators:
    print(f"     - {name}: {type(est).__name__}")
print(f"   Voting type: {model3.voting}")
print(f"   Classes: {classes}")
print("\n   ⚠️  ⚠️  ⚠️  WARNING ⚠️  ⚠️  ⚠️")
print("   This model does NOT support partial_fit() or warm_start!")
print("   Use ONLY with sample-based training (load data into RAM)")

✅ Approach 3 Model Ready: VotingClassifier Ensemble
   Models in ensemble: 4
     - logistic: LogisticRegression
     - decision_tree: DecisionTreeClassifier
     - random_forest: RandomForestClassifier
     - svm: SVC
   Voting type: soft
   Classes: [0 1]

   ⚠️  ⚠️  ⚠️  WARNING ⚠️  ⚠️  ⚠️
   This model does NOT support partial_fit() or warm_start!
   Use ONLY with sample-based training (load data into RAM)


#### 📋 Modeling Approach Selection Guide

| Cell | Model | Supports Chunked Training | RAM Usage | Accuracy Potential |
|:----:|:-----:|:-------------------------:|:---------:|:------------------:|
| **Option 1** | `SGDClassifier` | ✅ Yes (`partial_fit`) | 🟢 Low | 🟡 Good (75-80%) |
| **Option 2** | `HistGBC` | ✅ Yes (`warm_start`) | 🟡 Medium | 🟠 Better (80-85%) |
| **Option 3** | `Ensemble` | ❌ No (needs sample) | 🔴 High | 🟢 Best (85-90%) |

#### 🚀 My Recommendation

> **Use Option 1** - It works perfectly with your existing chunked training loop and requires no other code changes. Only switch to Option 2 or 3 if accuracy is insufficient.

##### Why Option 1?
- ✅ No modifications to your current training loop
- ✅ Memory efficient (works with 50k chunks)
- ✅ Fast training on large datasets
- ✅ Industry standard for credit scoring
- ✅ Easy to interpret and explain

##### When to Upgrade?
- **Switch to Option 2** if accuracy < 75% after Phase 1
- **Switch to Option 3** only if you need maximum accuracy AND have sufficient RAM (16GB+)

In [23]:
# ====================================================================================
# 3.3. Chunk preprocessing function
# ------------------------------------------------------------------------------------
# Cleans and prepares each chunk of data for training
# ====================================================================================

In [24]:
def preprocess_chunk(chunk: pd.DataFrame):
    """
    Clean a single chunk of loan data and prepare features/target.
    
    Parameters:
    -----------
    chunk : pd.DataFrame
        One chunk of data (50,000 rows) from the CSV file
    
    Returns:
    --------
    X : pd.DataFrame
        Features (loan_amnt, int_rate, annual_inc, dti, fico_range_low)
    y : pd.Series
        Target variable (0 = Good Loan, 1 = Charged Off)
    """
    
    # ------------------------------------------------------------------------------------
    # STEP 1: Clean the interest rate column
    # Problem: Interest rates come as strings with % (e.g., "10.5%")
    #          Also may have <NA>, NaN, or other missing values
    # Goal: Convert to numbers (e.g., 10.5) for machine learning
    # ------------------------------------------------------------------------------------
    
    # .astype(str) - Convert to string first (prevents crashes if already number)
    # .str.replace("<NA>", "") - Replace pandas missing value marker with empty string
    # .str.replace("nan", "") - Replace string 'nan' with empty string
    # .str.replace("None", "") - Replace 'None' with empty string
    # .str.replace(" ", "") - Remove any spaces
    # .str.rstrip("%") - Remove the % symbol from the end of each string
    # pd.to_numeric(errors="coerce") - Convert to number, turn invalid values to NaN
    # .astype("float32") - Convert to 32-bit floating point number
    chunk["int_rate"] = (
        chunk["int_rate"]
        .astype(str)              # Convert anything to string first
        .str.replace("<NA>", "")  # Replace pandas missing value marker
        .str.replace("nan", "")   # Replace 'nan' string
        .str.replace("None", "")  # Replace 'None' string
        .str.replace(" ", "")     # Remove any spaces
        .str.rstrip("%")          # Remove % symbol (e.g., "10.5%" -> "10.5")
    )
    
    # Convert empty strings to NaN, then to float32
    # errors="coerce" turns invalid values (like empty strings) into NaN
    chunk["int_rate"] = pd.to_numeric(chunk["int_rate"], errors="coerce").astype("float32")
    
    # ------------------------------------------------------------------------------------
    # STEP 2: Create binary target variable
    # Original: "loan_status" has values like "Fully Paid" or "Charged Off"
    # Goal: 0 = Good loan (Fully Paid), 1 = Bad loan (Charged Off)
    # ------------------------------------------------------------------------------------
    
    # Convert categorical to string temporarily to handle missing values
    # .astype(str) - Convert from category to string
    # .fillna("Unknown") - Replace NaN with "Unknown" (will be dropped later)
    # .str.replace("<NA>", "Unknown") - Replace pandas NA marker
    chunk["loan_status"] = (
        chunk["loan_status"]
        .astype(str)                    # Convert category to string (safe for fillna)
        .fillna("Unknown")              # Replace NaN with "Unknown"
        .str.replace("<NA>", "Unknown") # Replace pandas missing value marker
        .str.replace("nan", "Unknown")  # Replace 'nan' string
        .str.replace("None", "Unknown") # Replace 'None' string
    )
    
    # Now create binary target: True if "Charged Off", False otherwise
    # .astype("int8") - Converts True -> 1, False -> 0 (8-bit integer saves memory)
    chunk["loan_status"] = (
        chunk["loan_status"] == "Charged Off"
    ).astype("int8")
    
    # ------------------------------------------------------------------------------------
    # STEP 3: Remove rows with missing values (NaN)
    # Why: Machine learning models cannot handle missing values
    # ------------------------------------------------------------------------------------
    
    # Store original count for warning message
    original_len = len(chunk)
    
    # .dropna() - Removes ANY row that has a missing value (NaN) in ANY column
    chunk = chunk.dropna()
    
    # OPTIONAL: Warn if we lost too much data (more than 50%)
    # This helps detect data quality issues
    if len(chunk) < original_len * 0.5:  # If remaining rows < 50% of original
        print(f"⚠️ Warning: Dropped {original_len - len(chunk):,} rows ({100 - (len(chunk)/original_len*100):.1f}% of chunk)")
        print(f"   Remaining: {len(chunk):,} rows for training")
    
    # ------------------------------------------------------------------------------------
    # STEP 4: Separate features (X) from target (y)
    # Features (X) = Input variables used to make predictions
    # Target (y) = What we want to predict
    # ------------------------------------------------------------------------------------
    
    # Handle edge case: If no rows remain after dropping missing values
    if len(chunk) == 0:
        # Return empty DataFrame and Series to prevent errors
        return pd.DataFrame(), pd.Series(dtype="int8")
    
    # X = All columns EXCEPT the target column ("loan_status")
    # .drop(columns=["loan_status"]) - Removes target column from features
    X = chunk.drop(columns=["loan_status"])
    
    # y = Only the target column ("loan_status")
    # This is what the model will learn to predict
    y = chunk["loan_status"]
    
    # ------------------------------------------------------------------------------------
    # STEP 5: Return the prepared features and target
    # ------------------------------------------------------------------------------------
    return X, y


# ===================================================================
# TEST THE FUNCTION (Optional - for debugging)
# ===================================================================

# Uncomment these lines to test the function on a small sample
# test_chunk = pd.DataFrame({
#     "loan_amnt": [10000, 20000, None],
#     "int_rate": ["10.5%", "12.3%", "15.0%"],
#     "annual_inc": [50000, 60000, 70000],
#     "dti": [15.5, 20.0, 25.5],
#     "fico_range_low": [680, 700, None],
#     "loan_status": ["Fully Paid", "Charged Off", "Fully Paid"]
# })
# 
# print("Original chunk:")
# print(test_chunk)
# print("\nAfter preprocessing:")
# X_test, y_test = preprocess_chunk(test_chunk)
# print(f"Features (X):\n{X_test}")
# print(f"\nTarget (y):\n{y_test}")

In [25]:
# ====================================================================================
# 3.4. Streaming training loop
# ------------------------------------------------------------------------------------
# Works with many modeling approaches
# ====================================================================================

In [33]:
# Model Selection - Choose you approach here
# ===================================================================

# Uncomment ONLY ONE of the following options:

# OPTION 1: SGDClassifier (Logistic Regression) - RECOMMENDED
# Works with partial_fit, memory efficient, fast training
# model = SGDClassifier(loss="log_loss", random_state=42)
# training_method = "partial_fit"  # Uses partial_fit() for incremental learning

# OPTION 2: HistGradientBoostingClassifier - BETTER ACCURACY
# Works with warm_start, adds trees incrementally
# model = HistGradientBoostingClassifier(warm_start=True, max_iter=50, max_depth=10, random_state=42)
# training_method = "warm_start"  # Uses warm_start for incremental learning

# OPTION 3: Ensemble (Voting Classifier) - BEST ACCURACY (SAMPLE ONLY)
# Does NOT support incremental learning - use with sampled data only
# model = VotingClassifier([('logistic', LogisticRegression()), ('rf', RandomForestClassifier())])
# training_method = "batch"  # Uses standard fit() - NOT for chunked data

# Current Selection (Change this variable to switch approaches)
# ===================================================================

# Change this variable to switch models:
# Change this value to switch between models without editing code below
#training_method = "partial_fit"   # Approach 1: Fast & memory efficient
#training_method = "warm_start"  # Approach 2: Better accuracy
training_method = "batch"       # Approach 3: Best accuracy (sample only)

# Initialize model based on selected method
if training_method == "partial_fit":
    # OPTION 1: SGDClassifier for incremental learning
    
    model = model1  # Use pre-initialized model1
    
    print("✅ SELECTED: Phase 1 - SGDClassifier (Logistic Regression)")
    print("   Training method: partial_fit (incremental learning)")
    
elif training_method == "warm_start":
    # OPTION 2: HistGradientBoosting for better accuracy
    
    model = model2  # Use pre-initialized model2
    
    print("✅ SELECTED: Phase 2 - HistGradientBoostingClassifier")
    print("   Training method: warm_start (adds trees incrementally)")
    
elif training_method == "batch":
    # OPTION 3: Ensemble (NOT for chunked data - use with samples only)
    
    model = model3  # Use pre-initialized model3
    
    print("⚠️  WARNING: 'batch' method does NOT support incremental learning!")
    print("   This will NOT work with your chunked training loop.")
    print("   Use this only if you load all data into RAM at once.")
    print("\n   To use this method:")
    print("   1. Load sample: df = pd.read_csv(FILE_PATH, nrows=200000)")
    print("   2. Preprocess: X, y = preprocess_chunk(df)")
    print("   3. Train: model.fit(X, y)")
    print("\n   Press Ctrl+C to stop or wait 5 seconds to continue with Phase 1...")
    
    time.sleep(5)  # Give user time to read warning
    
    # Fall back to Phase 1
    model = model1  # Use pre-initialized model1
    training_method = "partial_fit"
    print("\n🔄 Falling back to Phase 1 - SGDClassifier")
    
else:
    # Default fallback to Phase 1 if invalid selection
    model = model1  # Use pre-initialized model1
    training_method = "partial_fit"
    print(f"⚠️ Unknown method '{training_method}'. Falling back to Phase 1")
    print("✅ SELECTED: Phase 1 - SGDClassifier")

# Classes Definition
# ===================================================================

# Possible target values (0 = Good Loan, 1 = Bad Loan/Charged Off)
# int8 uses only 1 byte per value (very memory efficient)
classes = np.array([0, 1], dtype=np.int8)

# Training Loop (Adapts based on selected method)
# ===================================================================

first_chunk = True   # Flag: first chunk needs class info (for partial_fit)
total_rows = 0       # Counter to track progress
current_iter = 50    # For warm_start: tracks number of trees (starts at 50)

print("\n" + "="*60)
print("STARTING STREAMING TRAINING")
print("="*60)

# Read CSV one chunk at a time (never loads entire file)
for chunk_num, chunk in enumerate(pd.read_csv(
    DATA_PATH,           # Path to the large CSV file
    usecols=USE_COLS,    # Only load these columns (ignores others)
    dtype=DTYPES,        # Use memory-efficient data types
    chunksize=CHUNK_SIZE,# Load 50,000 rows per iteration
    low_memory=True      # Optimize RAM usage while reading
), start=1):
    
    # Clean and prepare the current chunk
    X, y = preprocess_chunk(chunk)
    
    # Skip if all rows had missing values
    if len(X) == 0:
        print(f"⚠️ Chunk {chunk_num}: No valid rows after preprocessing (skipping)")
        continue
    
    # Train based on selected method
    # ===================================================================
    
    if training_method == "partial_fit":
        # METHOD 1: SGDClassifier incremental learning
        if first_chunk:
            # First chunk: must specify all possible classes
            model.partial_fit(X, y, classes=classes)
            first_chunk = False
        else:
            # Subsequent chunks: classes already known
            model.partial_fit(X, y)
            
    elif training_method == "warm_start":
        # METHOD 2: HistGradientBoosting with warm_start
        if first_chunk:
            # First chunk: initial training
            model.fit(X, y)
            first_chunk = False
        else:
            # Subsequent chunks: add more trees incrementally
            current_iter += 25  # Add 25 more trees each chunk
            model.set_params(max_iter=current_iter)
            model.fit(X, y)  # Continues training from where it left off
            
    elif training_method == "batch":
        # METHOD 3: Batch learning (NOT recommended for streaming)
        # This will retrain from scratch each chunk - very inefficient!
        model.fit(X, y)
    
    # Update and display progress
    total_rows += len(X)
    
    # Progress indicator with method-specific info
    if training_method == "partial_fit":
        print(f"✅ Chunk {chunk_num}: Processed {total_rows:>10,} total rows | Model: SGDClassifier")
    elif training_method == "warm_start":
        print(f"✅ Chunk {chunk_num}: Processed {total_rows:>10,} total rows | Trees: {current_iter} | Model: HistGBC")
    else:
        print(f"✅ Chunk {chunk_num}: Processed {total_rows:>10,} total rows | Model: {type(model).__name__}")

# Training Complete
# ===================================================================

print("\n" + "="*60)
print("TRAINING COMPLETED")
print("="*60)
print(f"📊 Total rows processed: {total_rows:,}")
print(f"🤖 Model used: {type(model).__name__}")
print(f"⚙️ Training method: {training_method}")
print(f"💾 Memory efficient: Yes (processed in {CHUNK_SIZE:,}-row chunks)")

if training_method == "warm_start":
    print(f"🌲 Final number of trees: {model.n_iter_}")
    
print("\n✅ Model is ready for predictions!")

⚠️  WARNING: 'batch' method does NOT support incremental learning!
   This will NOT work with your chunked training loop.
   Use this only if you load all data into RAM at once.

   To use this method:
   1. Load sample: df = pd.read_csv(FILE_PATH, nrows=200000)
   2. Preprocess: X, y = preprocess_chunk(df)
   3. Train: model.fit(X, y)

   Press Ctrl+C to stop or wait 5 seconds to continue with Phase 1...

🔄 Falling back to Phase 1 - SGDClassifier

STARTING STREAMING TRAINING
✅ Chunk 1: Processed     49,999 total rows | Model: SGDClassifier
✅ Chunk 2: Processed     99,998 total rows | Model: SGDClassifier
✅ Chunk 3: Processed    149,998 total rows | Model: SGDClassifier
✅ Chunk 4: Processed    199,998 total rows | Model: SGDClassifier
✅ Chunk 5: Processed    249,998 total rows | Model: SGDClassifier
✅ Chunk 6: Processed    299,998 total rows | Model: SGDClassifier
✅ Chunk 7: Processed    349,998 total rows | Model: SGDClassifier
✅ Chunk 8: Processed    399,998 total rows | Model: SGDCl

In [27]:
# ====================================================================================
# 4. Evaluating Model
# ====================================================================================

In [28]:
# ====================================================================================
# 4.1. Evaluating Model Accuracy
# ====================================================================================

In [29]:
# Load test data
TEST_SIZE = 50000
print(f"Loading {TEST_SIZE:,} test rows...")

for i, chunk in enumerate(pd.read_csv(
    DATA_PATH,
    usecols=USE_COLS,
    dtype=DTYPES,
    chunksize=TEST_SIZE,
    nrows=TEST_SIZE
)):
    X_test, y_test = preprocess_chunk(chunk)
    break

print(f"Test set size: {len(X_test):,} rows")
print(f"Default rate in test: {y_test.mean():.2%}")

# Make predictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Calculate metrics WITH zero_division=0 to suppress warnings
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)  # ← FIXED
recall = recall_score(y_test, y_pred, zero_division=0)        # ← FIXED
f1 = f1_score(y_test, y_pred, zero_division=0)               # ← FIXED
auc = roc_auc_score(y_test, y_proba)
cm = confusion_matrix(y_test, y_pred)

print("\n" + "="*60)
print("PERFORMANCE METRICS")
print("="*60)
print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall:    {recall:.4f} ({recall*100:.2f}%)")
print(f"F1 Score:  {f1:.4f} ({f1*100:.2f}%)")
print(f"AUC-ROC:   {auc:.4f} ({auc*100:.2f}%)")

print("\n" + "="*60)
print("CONFUSION MATRIX")
print("="*60)
print("                 Predicted")
print("                 Good   Bad")
print(f"Actual Good     {cm[0,0]:>6,}  {cm[0,1]:>6,}")
print(f"Actual Bad      {cm[1,0]:>6,}  {cm[1,1]:>6,}")

Loading 50,000 test rows...
Test set size: 49,999 rows
Default rate in test: 18.05%

PERFORMANCE METRICS
Accuracy:  0.7433 (74.33%)
Precision: 0.2798 (27.98%)
Recall:    0.2679 (26.79%)
F1 Score:  0.2737 (27.37%)
AUC-ROC:   0.5764 (57.64%)

CONFUSION MATRIX
                 Predicted
                 Good   Bad
Actual Good     34,747   6,225
Actual Bad       6,609   2,418


In [30]:
# ====================================================================================
# 4.1. Diagnosing Model (If needed)
# ====================================================================================

In [31]:
# ------------------------------------------------------------------------------------
# Check Model Coefficients
# ------------------------------------------------------------------------------------

#print("="*60)
#print("MODEL DIAGNOSTICS")
#print("="*60)
#
## Check if model has coefficients
#print(f"\nModel coefficients: {model.coef_}")
#print(f"Model intercept: {model.intercept_}")
#
## Check if coefficients are non-zero
#if np.all(model.coef_ == 0):
#    print("\n❌ PROBLEM: All coefficients are ZERO! Model did NOT learn.")
#    print("   Possible causes:")
#    print("   1. Learning rate too low")
#    print("   2. Regularization too strong (alpha too high)")
#    print("   3. No convergence (needs more iterations)")
#else:
#    print("\n✅ Model has non-zero coefficients - learning occurred")
#
## Check class weight setting
#print(f"\nClass weight setting: {model.class_weight}")
#
## Make a simple test prediction
#print("\n" + "="*60)
#print("SIMPLE TEST PREDICTION")
#print("="*60)
#
## Create a single test row with average values
#test_row = pd.DataFrame({
#    "loan_amnt": [15000],
#    "int_rate": [12.5],
#    "annual_inc": [65000],
#    "dti": [18.0],
#    "fico_range_low": [690]
#})
#
#pred = model.predict(test_row)
#proba = model.predict_proba(test_row)[:, 1]
#
#print(f"Test loan (typical values):")
#print(f"  Prediction: {'BAD' if pred[0] == 1 else 'GOOD'}")
#print(f"  Default probability: {proba[0]:.2%}")

MODEL DIAGNOSTICS


AttributeError: 'HistGradientBoostingClassifier' object has no attribute 'coef_'

In [ ]:
# ====================================================================================
# 5. Saving Model (For Later Use)
# ====================================================================================

In [34]:
# Create filename with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

In [35]:
model_filename = f"models/loan_default_model_{timestamp}.pkl"

# Save model
joblib.dump(model, model_filename)
print(f"✅ Model saved as: {model_filename}")

# Also save model info
model_info = {
    'model_type': type(model).__name__,
    'training_method': training_method,
    'total_rows_processed': total_rows,
    'chunk_size': CHUNK_SIZE,
    'timestamp': timestamp
}

print(f"\nModel Info:")
for key, value in model_info.items():
    print(f"  {key}: {value}")

✅ Model saved as: models/loan_default_model_20260410_173002.pkl

Model Info:
  model_type: SGDClassifier
  training_method: partial_fit
  total_rows_processed: 2647888
  chunk_size: 50000
  timestamp: 20260410_173002


In [36]:
# ====================================================================================
# 6. Loading and Testing Saved Model
# ====================================================================================

In [37]:
# ===========================================================
# LOAD AND TEST THE TRAINED MODEL
# ===========================================================

# -------------------------------------------------------------------
# STEP 1: Load the saved model from disk
# -------------------------------------------------------------------

# joblib.load() - Reads a saved model file and loads it back into memory
# model_filename - The name of the file where model was saved (e.g., "loan_model_20241201.pkl")
# loaded_model - Variable that now contains the trained model (ready to predict)
loaded_model = joblib.load(model_filename)

# print() - Displays message to confirm model loaded successfully
# f"..." - f-string allows inserting variables into text with {variable_name}
# ✅ - Green checkmark emoji for visual confirmation
print(f"✅ Model loaded: {model_filename}")

# -------------------------------------------------------------------
# STEP 2: Load a small sample of data for testing
# -------------------------------------------------------------------

# TEST_SAMPLE_SIZE - Number of rows to load from CSV (50,000 rows)
# 50,000 is small enough to fit in memory but large enough for reliable testing
TEST_SAMPLE_SIZE = 50000

# print() with \n - Adds a blank line before the message for better readability
# {TEST_SAMPLE_SIZE:,} - The :, adds commas to number (e.g., 50000 becomes 50,000)
print(f"\nLoading {TEST_SAMPLE_SIZE:,} rows for testing...")

# pd.read_csv() - Reads CSV file into pandas DataFrame
# Parameters:
#   DATA_PATH - The file path to your CSV file (e.g., r"D:\Data\...")
#   usecols=USE_COLS - Only load specific columns (saves memory, faster loading)
#   dtype=DTYPES - Use memory-efficient data types (float32 instead of float64)
#   nrows=TEST_SAMPLE_SIZE - Only load first 50,000 rows (not entire huge file)
new_data_sample = pd.read_csv(
    DATA_PATH,
    usecols=USE_COLS,
    dtype=DTYPES,
    nrows=TEST_SAMPLE_SIZE
)

# -------------------------------------------------------------------
# STEP 3: Preprocess the test data (clean and prepare)
# -------------------------------------------------------------------

# preprocess_chunk() - Our custom function that cleans the data
#   - Converts interest rate from "10.5%" to 10.5 (float)
#   - Converts loan_status to binary (0=Good, 1=Bad)
#   - Removes rows with missing values (NaN)
#   - Separates features (X) from target (y)
# 
# X_new - Features only (loan_amnt, int_rate, annual_inc, dti, fico_range_low)
# _ - Underscore means "ignore this value" (we don't need y for prediction)
X_new, _ = preprocess_chunk(new_data_sample)

# len(X_new) - Counts how many rows survived preprocessing
# {len(X_new):,} - Displays number with commas (e.g., 49,999)
print(f"After preprocessing: {len(X_new):,} rows")

# -------------------------------------------------------------------
# STEP 4: Make predictions on the test data
# -------------------------------------------------------------------

# if len(X_new) > 0 - Check if there is any data to predict (prevents errors)
if len(X_new) > 0:
    
    # loaded_model.predict() - Returns class predictions (0 = GOOD, 1 = BAD)
    # Parameters:
    #   X_new - Features of loans to predict
    # Returns: Array of 0s and 1s (e.g., [0, 0, 1, 0, 1, ...])
    predictions = loaded_model.predict(X_new)
    
    # loaded_model.predict_proba() - Returns probability scores for each class
    # Parameters:
    #   X_new - Features of loans to predict
    # Returns: Array of [probability_GOOD, probability_BAD] for each loan
    # [:, 1] - Selects only the BAD loan probabilities (index 1)
    # Example: [[0.85, 0.15], [0.30, 0.70]] -> [0.15, 0.70]
    probabilities = loaded_model.predict_proba(X_new)[:, 1]
    
    # -------------------------------------------------------------------
    # STEP 5: Create results DataFrame with predictions
    # -------------------------------------------------------------------
    
    # X_new.copy() - Creates a copy of the features (to avoid modifying original)
    results_df = X_new.copy()
    
    # results_df['Prediction'] - Adds new column with text labels
    # List comprehension: ['BAD' if p == 1 else 'GOOD' for p in predictions]
    #   Loops through each prediction (p)
    #   If p equals 1 -> 'BAD', otherwise -> 'GOOD'
    # Example: [0, 1, 0] becomes ['GOOD', 'BAD', 'GOOD']
    results_df['Prediction'] = ['BAD' if p == 1 else 'GOOD' for p in predictions]
    
    # results_df['Risk'] - Adds column with probability scores (0.00 to 1.00)
    results_df['Risk'] = probabilities
    
    # results_df['Risk_Level'] - Adds color-coded risk level based on probability
    # Logic:
    #   If Risk > 0.7    -> "🔴 HIGH" (red high emoji)
    #   If Risk > 0.3    -> "🟡 MEDIUM" (yellow medium emoji)
    #   Otherwise        -> "🟢 LOW" (green low emoji)
    results_df['Risk_Level'] = ['🔴 HIGH' if r > 0.7 else '🟡 MEDIUM' if r > 0.3 else '🟢 LOW' for r in probabilities]
    
    # -------------------------------------------------------------------
    # STEP 6: Display first 10 predictions
    # -------------------------------------------------------------------
    
    # print() with separator line - Creates visual separation
    # "="*70 - Creates a line of 70 equal signs
    print("\n" + "="*70)
    print("PREDICTIONS (First 10 Loans)")
    print("="*70)
    
    # for loop - Iterates through first 10 loans (or less if fewer than 10)
    # range(min(10, len(results_df))) - Takes minimum of 10 or actual row count
    # i - Loop counter (0, 1, 2, 3...)
    for i in range(min(10, len(results_df))):
        
        # results_df.iloc[i] - Gets row at position i (like row index)
        # iloc = integer location (access by position number)
        row = results_df.iloc[i]
        
        # print() with formatted output:
        #   f"..." - f-string for variable insertion
        #   {i+1} - Loan number (adds 1 because i starts at 0)
        #   ${row['loan_amnt']:,.0f} - Loan amount with $, commas, no decimals
        #   {row['fico_range_low']:.0f} - FICO score with no decimals
        #   {row['Prediction']} - GOOD or BAD
        #   {row['Risk']:.2%} - Risk as percentage with 2 decimals (e.g., 15.50%)
        #   {row['Risk_Level']} - HIGH/MEDIUM/LOW with emoji
        print(f"Loan {i+1}: ${row['loan_amnt']:,.0f} | FICO: {row['fico_range_low']:.0f} | "
              f"→ {row['Prediction']} | Risk: {row['Risk']:.2%} {row['Risk_Level']}")
    
    # -------------------------------------------------------------------
    # STEP 7: Print summary statistics
    # -------------------------------------------------------------------
    
    print("\n" + "="*70)
    print("SUMMARY")
    print("="*70)
    
    # len(predictions) - Total number of loans predicted
    print(f"Total loans: {len(predictions):,}")
    
    # (predictions == 0).sum() - Counts how many predictions are GOOD (0)
    # (predictions == 0).mean() - Percentage of GOOD loans (average of 0s and 1s)
    print(f"GOOD loans: {(predictions == 0).sum():,} ({(predictions == 0).mean():.1%})")
    
    # (predictions == 1).sum() - Counts how many predictions are BAD (1)
    # (predictions == 1).mean() - Percentage of BAD loans
    print(f"BAD loans:  {(predictions == 1).sum():,} ({(predictions == 1).mean():.1%})")
    
    # probabilities.mean() - Average risk score across all loans
    print(f"Avg risk: {probabilities.mean():.2%}")
    
    # -------------------------------------------------------------------
    # STEP 8: Save results to CSV file
    # -------------------------------------------------------------------
    
    # from datetime import datetime - Imports datetime module for timestamps
    from datetime import datetime
    
    # datetime.now() - Gets current date and time
    # .strftime('%Y%m%d_%H%M%S') - Formats timestamp as: YearMonthDay_HourMinuteSecond
    # Example: '20241201_143025' (Dec 1, 2024 at 2:30:25 PM)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # results_df.to_csv() - Saves DataFrame to CSV file
    # Parameters:
    #   f"loan_predictions_{timestamp}.csv" - Filename with timestamp
    #   index=False - Don't save row numbers (keeps file cleaner)
    results_df.to_csv(f"predictions/loan_predictions_{timestamp}.csv", index=False)
    
    # Confirm save with timestamp in filename
    print(f"\n✅ Saved to: predictions/loan_predictions_{timestamp}.csv")
    
# -------------------------------------------------------------------
# STEP 9: Handle case with no data
# -------------------------------------------------------------------

# else - Runs if len(X_new) == 0 (no rows survived preprocessing)
else:
    print("❌ No data to predict")

✅ Model loaded: models/loan_default_model_20260410_173002.pkl

Loading 50,000 rows for testing...
After preprocessing: 49,999 rows

PREDICTIONS (First 10 Loans)
Loan 1: $3,600 | FICO: 675 | → GOOD | Risk: 0.00% 🟢 LOW
Loan 2: $24,700 | FICO: 715 | → GOOD | Risk: 0.00% 🟢 LOW
Loan 3: $20,000 | FICO: 695 | → GOOD | Risk: 0.00% 🟢 LOW
Loan 4: $35,000 | FICO: 785 | → GOOD | Risk: 0.00% 🟢 LOW
Loan 5: $10,400 | FICO: 695 | → GOOD | Risk: 0.00% 🟢 LOW
Loan 6: $11,950 | FICO: 690 | → GOOD | Risk: 0.00% 🟢 LOW
Loan 7: $20,000 | FICO: 680 | → GOOD | Risk: 0.00% 🟢 LOW
Loan 8: $20,000 | FICO: 705 | → GOOD | Risk: 0.00% 🟢 LOW
Loan 9: $10,000 | FICO: 685 | → GOOD | Risk: 0.00% 🟢 LOW
Loan 10: $8,000 | FICO: 700 | → GOOD | Risk: 0.00% 🟢 LOW

SUMMARY
Total loans: 49,999
GOOD loans: 49,997 (100.0%)
BAD loans:  2 (0.0%)
Avg risk: 0.00%

✅ Saved to: predictions/loan_predictions_20260410_173006.csv
